# House Price Prediction : Pipeline

End-to-end sklearn pipeline for the Kaggle House Prices competition.
Final score(Best Submission) : 0.122 RMSE (log-scale) - Rank 469 / 4,106 (Top 12%)

Run cells top to bottom. Place train.csv and test.csv in the data/ folder before running.
Note: Optuna tuning is stochastic ( CV scores and ensemble weights will vary slightly between runs.)

In [1]:
#Imports

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
#Load data & remove outliers

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

test_ids = test["Id"]

train = train[~((train["GrLivArea"] > 4000) &
                (train["SalePrice"] < 200000))]

print("Shape after outlier removal:", train.shape)

y = np.log1p(train["SalePrice"])
X = train.drop(["SalePrice", "Id"], axis=1).reset_index(drop=True)
X_test = test.drop("Id", axis=1).reset_index(drop=True)
y = y.reset_index(drop=True)

Shape after outlier removal: (1458, 81)


## Preprocessing pipeline

Four custom sklearn transformers handle all preprocessing steps.
Each transformer learns statistics only from training data inside fit() - preventing data leakage.

In [4]:
#MissingValueHandler

class MissingValueHandler(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        X = X.copy()
        self.lot_frontage_medians_ = (
            X.groupby("Neighborhood")["LotFrontage"].median()
        )
        self.electrical_mode_  = X["Electrical"].mode()[0]
        self.mszoning_mode_    = X["MSZoning"].mode()[0]
        self.functional_mode_  = X["Functional"].mode()[0]
        self.utilities_mode_   = X["Utilities"].mode()[0]
        self.saletype_mode_    = X["SaleType"].mode()[0]
        self.exterior1_mode_   = X["Exterior1st"].mode()[0]
        self.exterior2_mode_   = X["Exterior2nd"].mode()[0]
        self.kitchenqual_mode_ = X["KitchenQual"].mode()[0]
        self.garagecars_med_   = X["GarageCars"].median()
        self.garagearea_med_   = X["GarageArea"].median()
        return self

    def transform(self, X, y=None):
        X = X.copy()
        X["PoolQC"]      = X["PoolQC"].fillna("NoPool")
        X["MiscFeature"] = X["MiscFeature"].fillna("None")
        X["Alley"]       = X["Alley"].fillna("None")
        X["Fence"]       = X["Fence"].fillna("None")
        X["MasVnrType"]  = X["MasVnrType"].fillna("None")
        X["FireplaceQu"] = X["FireplaceQu"].fillna("None")

        for col in ["GarageType","GarageFinish","GarageQual","GarageCond"]:
            X[col] = X[col].fillna("None")
        for col in ["BsmtQual","BsmtCond","BsmtExposure",
                    "BsmtFinType1","BsmtFinType2"]:
            X[col] = X[col].fillna("None")

        X["MasVnrArea"] = X["MasVnrArea"].fillna(0)
        X["GarageArea"] = X["GarageArea"].fillna(self.garagearea_med_)
        X["GarageCars"] = X["GarageCars"].fillna(self.garagecars_med_)

        for col in ["BsmtFinSF1","BsmtFinSF2","BsmtUnfSF",
                    "TotalBsmtSF","BsmtFullBath","BsmtHalfBath"]:
            X[col] = X[col].fillna(0)

        X["HasGarage"]   = X["GarageYrBlt"].notnull().astype(int)
        X["GarageYrBlt"] = X["GarageYrBlt"].fillna(0)

        def fill_lot_frontage(row):
            if pd.isnull(row["LotFrontage"]):
                nbhd = row["Neighborhood"]
                if nbhd in self.lot_frontage_medians_.index:
                    return self.lot_frontage_medians_[nbhd]
                else:
                    return self.lot_frontage_medians_.median()
            return row["LotFrontage"]

        X["LotFrontage"] = X.apply(fill_lot_frontage, axis=1)

        X["Electrical"]  = X["Electrical"].fillna(self.electrical_mode_)
        X["MSZoning"]    = X["MSZoning"].fillna(self.mszoning_mode_)
        X["Functional"]  = X["Functional"].fillna(self.functional_mode_)
        X["Utilities"]   = X["Utilities"].fillna(self.utilities_mode_)
        X["SaleType"]    = X["SaleType"].fillna(self.saletype_mode_)
        X["Exterior1st"] = X["Exterior1st"].fillna(self.exterior1_mode_)
        X["Exterior2nd"] = X["Exterior2nd"].fillna(self.exterior2_mode_)
        X["KitchenQual"] = X["KitchenQual"].fillna(self.kitchenqual_mode_)
        return X

In [5]:
#OrdinalEncoder

class OrdinalEncoder(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.mappings = {
            "Street":       {"Grvl":0, "Pave":1},
            "Utilities":    {"NoSeWa":0, "AllPub":1},
            "LandSlope":    {"Gtl":1, "Mod":2, "Sev":3},
            "ExterQual":    {"Fa":1, "TA":2, "Gd":3, "Ex":4},
            "ExterCond":    {"Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5},
            "BsmtQual":     {"None":0, "Fa":1, "TA":2, "Gd":3, "Ex":4},
            "BsmtCond":     {"None":0, "Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5},
            "BsmtExposure": {"None":0, "No":1, "Mn":2, "Av":3, "Gd":4},
            "BsmtFinType1": {"None":0, "Unf":1, "LwQ":2, "Rec":3,
                             "BLQ":4, "ALQ":5, "GLQ":6},
            "BsmtFinType2": {"None":0, "Unf":1, "LwQ":2, "Rec":3,
                             "BLQ":4, "ALQ":5, "GLQ":6},
            "HeatingQC":    {"Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5},
            "CentralAir":   {"N":0, "Y":1},
            "KitchenQual":  {"Fa":1, "TA":2, "Gd":3, "Ex":4},
            "Functional":   {"Sev":1, "Maj2":2, "Maj1":3, "Mod":4,
                             "Min2":5, "Min1":6, "Typ":7},
            "FireplaceQu":  {"None":0, "Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5},
            "GarageFinish": {"None":0, "Unf":1, "RFn":2, "Fin":3},
            "GarageQual":   {"None":0, "Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5},
            "GarageCond":   {"None":0, "Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5},
            "PavedDrive":   {"N":0, "P":1, "Y":2},
            "PoolQC":       {"NoPool":0, "Fa":1, "TA":2, "Gd":3, "Ex":4},
        }

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        X = X.copy()
        for col, mapping in self.mappings.items():
            if col in X.columns:
                X[col] = X[col].map(mapping)
        return X

In [6]:
#FeatureEngineering

class FeatureEngineer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        X = X.copy()
        X["TotalLivArea"]      = X["GrLivArea"] + X["TotalBsmtSF"]
        X["TotalBathrooms"]    = (X["FullBath"] + X["BsmtFullBath"] +
                                  X["HalfBath"] * 0.5 + X["BsmtHalfBath"] * 0.5)
        X["TotalSF"]           = X["TotalBsmtSF"] + X["1stFlrSF"] + X["2ndFlrSF"]
        X["TotalPorchSF"]      = (X["OpenPorchSF"] + X["3SsnPorch"] +
                                  X["EnclosedPorch"] + X["ScreenPorch"] + X["WoodDeckSF"])
        X["HouseAge"]          = X["YrSold"] - X["YearBuilt"]
        X["YearsSinceRemodel"] = X["YrSold"] - X["YearRemodAdd"]
        X["WasRemodelled"]     = (X["YearRemodAdd"] != X["YearBuilt"]).astype(int)
        X["IsNewHouse"]        = (X["YearBuilt"] == X["YrSold"]).astype(int)
        X["QualityXArea"]      = X["OverallQual"] * X["GrLivArea"]
        X["QualityXTotalSF"]   = X["OverallQual"] * X["TotalSF"]
        X["QualityXAge"]       = X["OverallQual"] * X["HouseAge"]
        X["ConditionXQuality"] = X["OverallCond"] * X["OverallQual"]
        X["HasGarage"]         = (X["GarageArea"] > 0).astype(int)
        X["HasBasement"]       = (X["TotalBsmtSF"] > 0).astype(int)
        X["HasFireplace"]      = (X["Fireplaces"] > 0).astype(int)
        X["HasPool"]           = (X["PoolArea"] > 0).astype(int)
        X["Has2ndFloor"]       = (X["2ndFlrSF"] > 0).astype(int)
        X["HasOutdoorSpace"]   = (X["TotalPorchSF"] > 0).astype(int)
        X["GarageScore"]       = X["GarageCars"] * X["GarageQual"]
        X["TotalBHK"]          = X["BedroomAbvGr"] + X["KitchenAbvGr"]
        X["LotRatio"]          = X["GrLivArea"] / X["LotArea"]
        return X

In [7]:
# CELL 6 — TargetEncoder

class TargetEncoder(BaseEstimator, TransformerMixin):

    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y):
        temp = X.copy()
        temp['_target'] = y.values
        self.encoding_maps_ = {}
        self.global_mean_ = temp['_target'].mean()
        for col in self.cols:
            self.encoding_maps_[col] = temp.groupby(col)['_target'].mean()
        return self

    def transform(self, X, y=None):
        X = X.copy()
        for col in self.cols:
            X[f'{col}_Encoded'] = (
                X[col].map(self.encoding_maps_[col]).fillna(self.global_mean_)
            )
            X = X.drop(columns=[col])
        return X

In [8]:
# OneHotEncodeNominal
# Neighborhood handled by TargetEncoder above

class OneHotEncodeNominal(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.nominal_cols = [
            "MSZoning", "LotShape", "LandContour", "LotConfig",
            # "Neighborhood" ← target encoded instead
            "Condition1", "Condition2", "BldgType",
            "HouseStyle", "RoofStyle", "RoofMatl", "Exterior1st",
            "Exterior2nd", "MasVnrType", "Foundation", "Heating",
            "Electrical", "GarageType", "Fence", "MiscFeature",
            "Alley", "SaleType", "SaleCondition"
        ]

    def fit(self, X, y=None):
        X_encoded = pd.get_dummies(X, columns=self.nominal_cols, drop_first=True)
        self.train_columns_ = X_encoded.columns.tolist()
        return self

    def transform(self, X, y=None):
        X = X.copy()
        X_encoded = pd.get_dummies(X, columns=self.nominal_cols, drop_first=True)
        X_encoded = X_encoded.reindex(columns=self.train_columns_, fill_value=0)
        return X_encoded

In [9]:
# Build pipeline
# Note: TargetEncoder runs outside pipeline (needs y)

preprocessing_pipeline = Pipeline([
    ("missing",  MissingValueHandler()),
    ("ordinal",  OrdinalEncoder()),
    ("features", FeatureEngineer()),
    ("onehot",   OneHotEncodeNominal()),
])

## Target encoding

Neighborhood is target encoded rather than one-hot encoded - replacing 25 sparse dummy columns
with a single mean SalePrice per neighborhood. Out-of-fold encoding prevents leakage:
each row's encoding is computed from folds it was never part of.

In [10]:
# Run pipeline + OOF target encoding

TARGET_ENCODE_COLS = ['Neighborhood']  #learnt from previous submissions

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("Fitting preprocessing pipeline...")
X_processed      = preprocessing_pipeline.fit_transform(X)
X_test_processed = preprocessing_pipeline.transform(X_test)

print("Running OOF target encoding...")
oof_arrays = {col: np.zeros(len(X)) for col in TARGET_ENCODE_COLS}

for fold, (train_idx, val_idx) in enumerate(kf.split(X_processed)):
    X_tr_raw  = X.iloc[train_idx]
    X_val_raw = X.iloc[val_idx]
    y_tr      = y.iloc[train_idx]

    te = TargetEncoder(cols=TARGET_ENCODE_COLS)
    te.fit(X_tr_raw, y_tr)
    X_val_encoded = te.transform(X_val_raw)

    for col in TARGET_ENCODE_COLS:
        oof_arrays[col][val_idx] = X_val_encoded[f'{col}_Encoded'].values

print("OOF encoding done.")

# Add to training data, drop original column
X_processed_df = pd.DataFrame(X_processed)
for col in TARGET_ENCODE_COLS:
    X_processed_df[f'{col}_Encoded'] = oof_arrays[col]
    if col in X_processed_df.columns:
        X_processed_df = X_processed_df.drop(columns=[col])

# Test set — fit on ALL training data (no leakage risk)
te_full = TargetEncoder(cols=TARGET_ENCODE_COLS)
te_full.fit(X, y)
X_test_encoded = te_full.transform(X_test)

X_test_processed_df = pd.DataFrame(X_test_processed)
for col in TARGET_ENCODE_COLS:
    X_test_processed_df[f'{col}_Encoded'] = X_test_encoded[f'{col}_Encoded'].values
    if col in X_test_processed_df.columns:
        X_test_processed_df = X_test_processed_df.drop(columns=[col])

# Sanity check
print("\nAny object columns remaining?")
print(X_processed_df.dtypes[X_processed_df.dtypes == 'object'])
print("Train shape:", X_processed_df.shape)
print("Test shape: ", X_test_processed_df.shape)
print("Shapes match:", X_processed_df.shape[1] == X_test_processed_df.shape[1])

Fitting preprocessing pipeline...
Running OOF target encoding...
OOF encoding done.

Any object columns remaining?
Series([], dtype: object)
Train shape: (1458, 204)
Test shape:  (1459, 204)
Shapes match: True


In [11]:
# Tune XGBoost with Optuna

def objective_xgb(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":        trial.suggest_int("max_depth", 3, 9),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
        "random_state": 42
    }
    scores = cross_val_score(
        XGBRegressor(**params), X_processed_df, y,
        cv=kf, scoring="neg_root_mean_squared_error"
    )
    return -scores.mean()

print("Tuning XGBoost...")
study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(objective_xgb, n_trials=100, show_progress_bar=True)
print(f"XGBoost best RMSE: {study_xgb.best_value:.4f}")

Tuning XGBoost...


  0%|          | 0/100 [00:00<?, ?it/s]

XGBoost best RMSE: 0.1152


## Hyperparameter tuning

XGBoost and LightGBM tuned independently with Optuna (100 trials each, 5-fold CV).

In [12]:
# Tune LightGBM with Optuna

def objective_lgb(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":         trial.suggest_int("max_depth", 3, 9),
        "num_leaves":        trial.suggest_int("num_leaves", 20, 150),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
        "random_state": 42,
        "verbose": -1
    }
    scores = cross_val_score(
        lgb.LGBMRegressor(**params), X_processed_df, y,
        cv=kf, scoring="neg_root_mean_squared_error"
    )
    return -scores.mean()

print("Tuning LightGBM...")
study_lgb = optuna.create_study(direction="minimize")
study_lgb.optimize(objective_lgb, n_trials=100, show_progress_bar=True)
print(f"LightGBM best RMSE: {study_lgb.best_value:.4f}")

Tuning LightGBM...


  0%|          | 0/100 [00:00<?, ?it/s]

LightGBM best RMSE: 0.1160


## Final model and submission

Models trained on full training data. Blended by inverse RMSE weight : better CV score = higher weight.
Predictions reverse-transformed with np.expm1 before submission.

In [13]:
# Train final models & generate submission

print("\nTraining final models...")

xgb_final = XGBRegressor(**study_xgb.best_params, random_state=42)
xgb_final.fit(X_processed_df, y)

lgb_final = lgb.LGBMRegressor(**study_lgb.best_params, random_state=42, verbose=-1)
lgb_final.fit(X_processed_df, y)

xgb_rmse = study_xgb.best_value
lgb_rmse = study_lgb.best_value

xgb_w = (1/xgb_rmse) / ((1/xgb_rmse) + (1/lgb_rmse))
lgb_w = (1/lgb_rmse) / ((1/xgb_rmse) + (1/lgb_rmse))

print(f"Ensemble weights — XGBoost: {xgb_w:.3f}  LightGBM: {lgb_w:.3f}")

preds = (xgb_w * xgb_final.predict(X_test_processed_df) +
         lgb_w * lgb_final.predict(X_test_processed_df))

submission = pd.DataFrame({
    "Id":        test_ids,
    "SalePrice": np.expm1(preds)
})
submission.to_csv("submission_FINAL.csv", index=False)

print(f"\nXGBoost CV RMSE : {xgb_rmse:.4f}")
print(f"LightGBM CV RMSE: {lgb_rmse:.4f}")
print(f"Submission saved — shape: {submission.shape}")
print(submission.head())


Training final models...
Ensemble weights — XGBoost: 0.502  LightGBM: 0.498

XGBoost CV RMSE : 0.1152
LightGBM CV RMSE: 0.1160
Submission saved — shape: (1459, 2)
     Id      SalePrice
0  1461  124236.706488
1  1462  158489.572171
2  1463  179306.126742
3  1464  193972.040178
4  1465  183314.819441
